In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_118_DTU_Delhi_CPCB_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,71.18,288.33,44.67,24.09,68.76,52.90,21.92,0.74,19.30,...,0.28,NaN,86.98,0.43,87.98,NaN,0.0,NaN,NaN,NaN
1,2024-01-02,76.61,290.95,47.35,26.95,74.31,57.40,22.19,0.64,20.15,...,NaN,NaN,80.49,0.46,62.49,NaN,0.0,NaN,NaN,NaN
2,2024-01-03,81.94,308.65,52.09,26.41,77.62,61.41,22.01,1.03,19.65,...,0.67,NaN,90.27,0.58,51.68,NaN,0.0,NaN,NaN,NaN
3,2024-01-04,86.58,349.36,60.52,21.61,80.82,65.78,22.18,1.02,19.19,...,0.57,NaN,93.07,0.36,182.41,NaN,0.0,NaN,NaN,NaN
4,2024-01-05,64.61,267.06,48.99,19.80,68.79,54.30,22.36,0.85,19.28,...,0.48,NaN,93.11,0.45,106.55,NaN,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,58.43,249.17,25.84,41.69,67.47,47.41,65.89,0.99,36.25,...,0.11,NaN,93.53,2.49,184.82,0.0,0.0,10.61,NaN,NaN
362,2024-12-28,33.83,150.77,28.76,33.31,61.62,41.27,66.14,0.79,35.19,...,0.19,NaN,96.43,0.78,236.25,0.0,0.0,35.96,NaN,NaN
363,2024-12-29,31.21,139.75,19.58,27.01,46.60,25.91,65.94,0.57,35.88,...,0.12,NaN,92.83,1.67,275.89,0.0,0.0,161.44,NaN,NaN
364,2024-12-30,38.01,152.83,17.34,29.12,46.45,41.57,65.16,0.55,38.38,...,0.12,NaN,87.79,1.19,272.54,0.0,0.0,157.32,NaN,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Toluene (µg/m³)', 'BP (mmHg)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp              0
PM2.5 (µg/m³)          0
PM10 (µg/m³)           0
NO (µg/m³)             0
NO2 (µg/m³)            0
NOx (ppb)              0
NH3 (µg/m³)            0
SO2 (µg/m³)            0
CO (mg/m³)             0
Ozone (µg/m³)          0
Benzene (µg/m³)        0
Eth-Benzene (µg/m³)    0
MP-Xylene (µg/m³)      0
RH (%)                 0
WS (m/s)               0
WD (deg)               0
RF (mm)                0
TOT-RF (mm)            0
SR (W/mt2)             0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 19)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01          71.18        288.33       44.67        24.09   
1  2024-01-02          76.61        290.95       47.35        26.95   
2  2024-01-03          81.94        308.65       52.09        26.41   
3  2024-01-04          86.58        349.36       60.52        21.61   
4  2024-01-05          64.61        267.06       48.99        19.80   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      68.76        52.90        21.92        0.74          19.30   
1      74.31        57.40        22.19        0.64          20.15   
2      77.62        61.41        22.01        1.03          19.65   
3      80.82        65.78        22.18        1.02          19.19   
4      68.79        54.30        22.36        0.85          19.28   

   Benzene (µg/m³)  Eth-Benzene (µg/m³)  MP-Xylene (µg/m³)  RH (%)  WS (m/s)  \
0             0.14                 0.61              0.

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Eth-Benzene (µg/m³),MP-Xylene (µg/m³),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2)
0,2024-01-01,-0.007827,0.549580,0.956687,-0.881362,-0.141478,-0.354856,-1.188391,-0.231311,-1.578455,-2.017011,-0.139174,0.603544,1.183665,-1.157510,-1.343295,0.0,0.0,0.041428
1,2024-01-02,0.123872,0.571437,1.203649,-0.669458,0.118431,-0.103193,-1.166703,-0.577568,-1.477622,-1.647687,-0.139174,-0.190042,0.755813,-1.113209,-1.723958,0.0,0.0,0.041428
2,2024-01-03,0.253145,0.719093,1.640440,-0.709468,0.273440,0.121067,-1.181161,0.772835,-1.536936,-1.542166,-0.139174,-0.190042,1.400557,-0.936006,-1.885393,0.0,0.0,0.041428
3,2024-01-04,0.365683,1.058703,2.417265,-1.065110,0.423297,0.365460,-1.167506,0.738209,-1.591504,-1.489405,-0.139174,3.311073,1.585147,-1.260878,0.066907,0.0,0.0,0.041428
4,2024-01-05,-0.167175,0.372142,1.354775,-1.199217,-0.140073,-0.276561,-1.153047,0.149572,-1.580828,-1.964250,-0.139174,2.470805,1.587784,-1.127976,-1.065974,0.0,0.0,0.041428
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,-0.317064,0.222901,-0.778498,0.422659,-0.201889,-0.661885,2.343571,0.634332,0.432268,0.937581,4.022038,-0.983629,1.615472,1.884479,0.102898,0.0,0.0,0.041428
362,2024-12-28,-0.913709,-0.597969,-0.509420,-0.198233,-0.475847,-1.005266,-0.101571,-0.058183,0.306524,0.937581,3.316118,-0.236724,1.806654,-0.640667,0.870945,0.0,0.0,0.041428
363,2024-12-29,-0.977254,-0.689900,-1.355357,-0.665013,-1.179240,-1.864277,2.347588,-0.819948,0.388376,1.095863,2.238661,-0.890265,1.569325,0.673590,1.462922,0.0,0.0,-1.663661
364,2024-12-30,-0.812328,-0.580784,-1.561773,-0.508679,-1.186265,-0.988488,2.284933,-0.889200,0.684943,1.148624,2.052893,-0.890265,1.237064,-0.035223,1.412894,0.0,0.0,-1.709503


In [10]:
df.to_excel('DTUDelhi2024.xlsx', index=False)